In [50]:
import json
import random
from pathlib import Path

ROOT = Path("../")

def evaluate_policy(file_path, N=10000, max_steps=100, delays=(0, 1)):
    # ===== LOAD =====
    with open(file_path) as f:
        data = json.load(f)

    graph = data["graph"]
    policy = data["policy"]

    adj = graph["adj_matrix"]
    attack_duration = graph["attack_duration"]
    num_nodes = len(adj)

    # ===== HELPERS =====
    def sample_action(dist):
        actions = list(map(int, dist.keys()))
        probs = list(dist.values())
        return random.choices(actions, weights=probs)[0]

    def get_policy(player, def_hist):
        key = f"p={player}|def_hist=" + ",".join(map(str, def_hist))
        return policy.get(key, None)

    # ===== EPISODE =====
    def run_episode(use_trained_attacker=True):
        defender_pos = random.randint(0, num_nodes - 1)
        delay = random.choice(delays)

        def_hist = [defender_pos]

        attacker_target = None
        attack_timer = None

        for t in range(max_steps):

            # attacker
            if t == delay:
                if use_trained_attacker:
                    pol = get_policy(1, def_hist)
                    attacker_target = sample_action(pol) if pol else random.randint(0, num_nodes - 1)
                else:
                    attacker_target = random.randint(0, num_nodes - 1)

                attack_timer = attack_duration[attacker_target]

            # defender
            pol = get_policy(0, def_hist)
            next_pos = sample_action(pol) if pol else random.randint(0, num_nodes - 1)

            defender_pos = next_pos
            def_hist.append(defender_pos)

            # outcome
            if attacker_target is not None:
                if defender_pos == attacker_target:
                    return 1
                cost = adj[def_hist[-2]][def_hist[-1]]
                attack_timer -= cost
                if attack_timer <= 0:
                    return 0

        return 0

    # ===== RUN =====
    results = {}

    for mode in [True, False]:
        captures = sum(run_episode(use_trained_attacker=mode) for _ in range(N))
        fails = N - captures

        key = "trained_attacker" if mode else "random_attacker"
        results[key] = {
            "capture": captures / N,
            "fail": fails / N
        }

        print("\n=== MODE:", key.upper(), "===")
        print(f"Rollouts: {N}")
        print(f"Capture: {captures} ({captures/N:.3f})")
        print(f"Fail:    {fails} ({fails/N:.3f})")

    return results

In [51]:
file_path = ROOT / "results/full/gdynia_targets_3.json"

evaluate_policy(str(file_path), N=100000)


=== MODE: TRAINED_ATTACKER ===
Rollouts: 100000
Capture: 11015 (0.110)
Fail:    88985 (0.890)

=== MODE: RANDOM_ATTACKER ===
Rollouts: 100000
Capture: 29075 (0.291)
Fail:    70925 (0.709)


{'trained_attacker': {'capture': 0.11015, 'fail': 0.88985},
 'random_attacker': {'capture': 0.29075, 'fail': 0.70925}}

In [52]:
file_path = ROOT / "results/full/gdynia_targets_4.json"

evaluate_policy(str(file_path), N=100000)


=== MODE: TRAINED_ATTACKER ===
Rollouts: 100000
Capture: 27554 (0.276)
Fail:    72446 (0.724)

=== MODE: RANDOM_ATTACKER ===
Rollouts: 100000
Capture: 37399 (0.374)
Fail:    62601 (0.626)


{'trained_attacker': {'capture': 0.27554, 'fail': 0.72446},
 'random_attacker': {'capture': 0.37399, 'fail': 0.62601}}

In [53]:
file_path = ROOT / "results/full/san_francisko_8.json"

evaluate_policy(str(file_path), N=100000)


=== MODE: TRAINED_ATTACKER ===
Rollouts: 100000
Capture: 37230 (0.372)
Fail:    62770 (0.628)

=== MODE: RANDOM_ATTACKER ===
Rollouts: 100000
Capture: 37747 (0.377)
Fail:    62253 (0.623)


{'trained_attacker': {'capture': 0.3723, 'fail': 0.6277},
 'random_attacker': {'capture': 0.37747, 'fail': 0.62253}}

In [55]:
file_path = ROOT / "results/full/star_shield.json"

evaluate_policy(str(file_path), N=100000)


=== MODE: TRAINED_ATTACKER ===
Rollouts: 100000
Capture: 37356 (0.374)
Fail:    62644 (0.626)

=== MODE: RANDOM_ATTACKER ===
Rollouts: 100000
Capture: 44722 (0.447)
Fail:    55278 (0.553)


{'trained_attacker': {'capture': 0.37356, 'fail': 0.62644},
 'random_attacker': {'capture': 0.44722, 'fail': 0.55278}}